# Industrial AI Capability Building Workshop
## Hands-On Notebook: Predicting Filling-Line Reject Rate with XGBoost & Optuna

---
### The Problem

Every carton the Serac filler rejects is finished product thrown away — milk, carton,
straw, line time and energy. Today, reject rate is only known *after* the shift from
the OEE report. By then, the loss is already sunk.

**Goal:** Predict shift reject rate from operating conditions (line speed, downtime, fill
variability, changeover, CIP) using **XGBoost fine-tuned with Optuna Bayesian optimization**,
and explain which levers drive high rejects using **SHAP**.

---

## 0. Configuration

**The main cell you change to reuse this notebook on a similar problem.** Point it at a
different sheet, list the inputs, name the target, and set `problem_type` and tuning settings.
Everything below adapts automatically.

### Prompt 0: Project Configuration
**Role:** Industrial AI Architect  
**Context:** Central configuration dictionary defining data source, target, and feature sets for the pipeline.  
**Task:**  
1. Create a `CONFIG` dictionary containing:
   - `data_path`: (https://raw.githubusercontent.com/dissabnd/Industry_AI_workshop/refs/heads/main/data/Filling_Machine_Mock_Training_Data.csv)'
   - `problem_type`: "regression"
   - `target`: "reject_rate_pct"
   - `numeric_features` (partitioned into two groups):
     - *Raw Telemetry (from CSV):* `Pack Size (ml)`, `Run Time (min)`, `Planned Downtime (min)`, `Unplanned Downtime (min)`, `Changeover Time (min)`, `CIP Time (min)`, `Target Speed (packs/hr)`, `Actual Speed (packs/hr)`, `Fill Volume Avg (ml)`, `Fill Volume Std Dev (ml)`
     - *Derived Stress Features (calculated in Section 1):* `speed_shortfall`, `speed_ratio`, `total_downtime`, `unplanned_downtime_ratio`, `fill_volume_cv`
   - `categorical_features`: `["Product / SKU", "Container", "Shift"]`
   - Model tuning options: `model_type` ("xgboost"), `use_optuna` (True), `optuna_trials` (30), `cv_folds` (5).


## Setup

### Prompt: Environment Setup
**Role:** Python ML Engineer  
**Context:** Environment setup for local Jupyter or Google Colab execution.  
**Task:**  
1. Import standard libraries: numpy, pandas, matplotlib.pyplot, and IPython.display.
2. Import Scikit-learn modules for data splitting, preprocessing pipelines, metrics, and models.
3. Import xgboost, optuna, and shap (with automatic fallback installation if missing).
4. Set pandas display limit to 60 columns and matplotlib default figure size to (8, 4).


## 1. Load the data

### Prompt 1: Data Ingestion & Stress Feature Engineering
**Role:** Process & Feature Engineer  
**Context:** Shift-level telemetry dataset from a high-speed packaging line (1,095 shifts).  
**Task:**  
1. Load the CSV from `CONFIG['data_path']` and parse `Date` to datetime.
2. Create a binary `has_fault` flag (1 if `Main Fault Code` is present, 0 otherwise).
3. Convert fractional `Reject Rate (%)` to percentage points (`reject_rate_pct = Reject Rate (%) * 100`).
4. Calculate 5 domain stress and stability features:
   - `speed_shortfall` = Target Speed - Actual Speed
   - `speed_ratio` = Actual Speed / Target Speed
   - `total_downtime` = Planned Downtime + Unplanned Downtime
   - `unplanned_downtime_ratio` = Unplanned Downtime / total_downtime
   - `fill_volume_cv` = Fill Volume Std Dev / Fill Volume Avg
5. Print date range, total rows, and preview `.head()` for key columns.  
**Constraints:** Add epsilon (1e-5) in denominators to prevent division by zero.


## 2. Anchor: what does the reject rate actually look like?

Let's plot the Y variable.  Two views: its trend over the year, and its distribution. This makes the problem concrete before a single model is trained.

### Prompt 2: Target Visualization & Financial Anchor
**Role:** Plant Operations & Financial Analyst  
**Context:** Visual problem anchor showing historical reject rates, baseline distribution, and annual scrap cost.  
**Task:**  
1. Create a 1x2 figure (width ratios [2, 1]):
   - (a) Daily average reject rate trend over time with a 14-day rolling average and annual mean reference line.
   - (b) Histogram distribution of shift reject rates with the annual mean reference line.
2. Calculate and print total packs produced, total packs scrapped, annual mean reject rate, and the volume prize (packs/year) for every 0.1 percentage point reduction.  
**Constraints:** Use corporate palette (#4C72B0, #C44E52, #B0B0B0) and format pack counts with commas.


## 3. Explore: what moves the reject rate?

### Prompt 3: Exploratory Data Analysis
**Role:** Data Quality & Analytics Specialist  
**Context:** Baseline data audit, statistical summary, and visual exploration across operational drivers.  
**Task:**  
1. Display transposed descriptive statistics (`describe().T`) and missing value counts for all features and target.
2. Create a 1x3 categorical bar chart comparing mean reject rate by: (1) `Product / SKU` (sorted), (2) `Shift`, and (3) `has_fault` (clean vs faulted shifts). add standard error. 
3. Create a 1x2 numeric figure showing: (a) scatter plot of `Unplanned Downtime (min)` vs `reject_rate_pct`, and (b) sorted horizontal bar chart of Pearson correlations between numeric inputs and target.


## 4. A word on target leakage (important)

Some columns in this workbook are **not allowed as inputs**, even though they're strongly
related to the target. `Reject Packs`, `Good Packs`, `Total Packs`, `Quality (%)`  and
`OEE (%)` are all *calculated from the reject count itself*. Feeding them to the model would
produce a near-perfect score that collapses the moment you deploy — because in real time you
wouldn't know them until the rejects had already happened. That is **target leakage**.

We also **removed `has_fault`** — a binary flag for whether a fault occurred. This is pure
leakage: you can't know a fault will happen until it happens. While it's strongly predictive
of rejects, using it collapses the model into 'if fault then high rejects' — unhelpful for
prevention. The real drivers are the underlying conditions (speed shortfall, fill drift) that
*precede* the fault.

The features we kept are the operating **conditions** of the shift (product, speed, fill, downtime,
changeover, CIP). Note: `Unplanned Downtime` and `Run Time` are known at *end of shift*, so
this model is best used for **post-shift explanation** ("why did that shift reject high?") not
strict pre-shift forecasting. For a true predictive model, remove those two and retrain.



## 5. Clean the data

Remove duplicated rows and null out impossible sensor spikes; genuinely missing readings are
imputed inside the pipeline so the same rules apply to any future data.

### Prompt 5: Data Cleansing & Outlier Filtering
**Role:** Data Cleansing Engineer  
**Context:** Removing duplicates and handling sensor spikes.  
**Task:**  
1. Remove duplicate rows from a copy of `df_raw` and reset index.
2. For each continuous numeric feature, calculate IQR and set outlier bounds at $[Q1 - 3\times IQR, Q3 + 3\times IQR]$.
3. Null out extreme spikes outside these bounds to `np.nan` for pipeline median imputation.
4. Print count of nulled values per feature and total clean rows.  
**Constraints:** Skip binary indicator columns.


## 6. Train / test split + preprocessing pipeline

### Prompt 6: Train/Test Split & Pipeline Construction
**Role:** ML Pipeline Architect  
**Context:** Automated, leak-free data partitioning and preprocessing.  
**Task:**  
1. Split features and target into 75% train and 25% test sets (`random_state=42`).
2. Build a `ColumnTransformer` containing:
   - Numeric: Median SimpleImputer + StandardScaler.
   - Categorical: Most frequent SimpleImputer + OneHotEncoder (`handle_unknown='ignore'`).
3. Print train and test set sizes.


## 7. Model Training & Hyperparameter Fine-Tuning with Optuna

### Prompt 7: XGBoost Training & Optuna Tuning
**Role:** Machine Learning Engineer  
**Context:** Optimizing XGBoost hyperparameters using Bayesian optimization to maximize cross-validation R² score.  
**Task:**  
1. Preprocess train and test data with the ColumnTransformer and extract feature names.
2. Run an Optuna study (30 trials, 5-fold CV) optimizing R² over tree depth, estimators, learning rate, subsampling, and regularization.
3. Print the best CV score and optimal parameters.
4. Fit the final XGBoost model on the full training set using the best parameters.  
**Constraints:** Set Optuna logging to WARNING. Set `random_state=42` and `n_jobs=-1` for reproducibility.


## 8. Evaluate on the held-out test set

With XGBoost and Optuna tuning, the model captures non-linear interactions across downtime, speed shortfalls, and fill variability to deliver high predictive accuracy on unseen test shifts.

### Prompt 8: Model Evaluation & Diagnostics
**Role:** Model Validation Lead  
**Context:** Evaluating model accuracy on unseen test data.  
**Task:**  
1. Generate predictions for the test set.
2. Calculate and print MAE, RMSE, and R² score.
3. Plot Predicted vs. Actual plot and residual plot.


## 9. Explainable AI (XAI) with SHAP

SHAP (SHapley Additive exPlanations) provides both **global operational insights** (which parameters drive rejects and in what direction) and **local shift diagnostics** (root-cause explanations for specific high-reject runs).


### 9a. Global Feature Impact: SHAP Beeswarm Plot

### Prompt 9a: SHAP Beeswarm Summary Plot
**Role:** Explainable AI (XAI) Lead  
**Context:** Explaining global feature impact distributions across test shifts using Shapley values.  
**Task:**  
1. Instantiate a `shap.TreeExplainer` on the trained XGBoost model.
2. Calculate SHAP values for test data and assign feature names.
3. Render a SHAP beeswarm summary plot for the top 12 features.


### 9b. Top Driver Response Curves: SHAP Dependence Plots

### Prompt 9b: SHAP Dependence Plots for Top 4 Drivers
**Role:** Process Optimization Engineer  
**Context:** Inspecting non-linear response curves and operating thresholds for top drivers.  
**Task:**  
1. Extract the top 4 drivers based on mean absolute SHAP values.
2. Create a 2x2 grid of SHAP dependence scatter plots with a horizontal zero reference line ($y=0$).
3. Print operational interpretation guidelines.


### 9c. Individual Shift Diagnostics: SHAP Waterfall Plot

### Prompt 9c: Individual Shift Diagnostics (SHAP Waterfall)
**Role:** Plant Shift Supervisor  
**Context:** Explaining a single high-reject shift for root-cause diagnosis.  
**Task:**  
1. Identify the test shift with the highest predicted reject rate (`np.argmax(pred)`).
2. Render a SHAP waterfall plot explaining how each feature contributed to the prediction.
3. Display the raw shift telemetry data.


## 10. Reusing this notebook

To retarget this exact workflow, edit **only** the `CONFIG` cell:

- Different regression target (e.g. `Unplanned Downtime (min)`, `Fill Volume Std Dev (ml)`):
  change `target` and the feature lists.
- A **classification** problem (e.g. predict `has_fault`): set `problem_type` to
  `"classification"` and point `target` at a 0/1 column — the evaluation, importance and SHAP
  sections switch to confusion-matrix / class outputs automatically.
